# 02 模型评测

**目标**：调用模型 API，收集每条样本的回答，输出 `results/raw/raw_results.csv`

**建议顺序**：
1. 先跑 DeepSeek（便宜，验证流程）
2. 流程通后再加 Kimi / Qwen-Long

⚠️ **前置条件**：已在 `.env` 文件中配置 API Key

In [13]:
import sys
sys.path.insert(0, '..')

import os
from pathlib import Path
import pandas as pd
import yaml
from dotenv import load_dotenv

load_dotenv('../.env')

from src.eval_runner import run_eval, call_model, get_client

config = yaml.safe_load(open('../configs/eval_config.yaml', encoding='utf-8'))
print('配置加载成功')

配置加载成功


## Step 1：检查 API Key 配置

In [14]:
key_map = {
    'deepseek': 'DEEPSEEK_API_KEY',
    'kimi':     'MOONSHOT_API_KEY',
    'qwen':     'DASHSCOPE_API_KEY',
}
print('API Key 状态：')
for model, env_var in key_map.items():
    val = os.getenv(env_var, '')
    status = '✅ 已配置' if val and not val.startswith('sk-xxx') else '❌ 未配置'
    masked = (val[:8] + '...' + val[-4:]) if len(val) > 12 else '（空）'
    print(f'  {model:12s} ({env_var}): {status}  {masked}')

API Key 状态：
  deepseek     (DEEPSEEK_API_KEY): ✅ 已配置  sk-5cfdc...a922
  kimi         (MOONSHOT_API_KEY): ✅ 已配置  sk-sExnE...ifpB
  qwen         (DASHSCOPE_API_KEY): ✅ 已配置  sk-a1bce...10e4


## Step 2：单条冒烟测试（验证 API 连通性）

In [15]:
TEST_MODEL = 'deepseek'  # 改为你想测试的模型

try:
    client, model_name = get_client(TEST_MODEL, config)
    test_context = '本公司于2024年1月1日在北京成立，注册资本为1000万元人民币，法人代表为张三。'
    test_question = '公司的注册资本是多少？'
    
    response, tokens, latency = call_model(
        client, model_name,
        context=test_context,
        question=test_question,
        max_tokens=64,
    )
    print(f'✅ [{TEST_MODEL}] 连通性测试通过')
    print(f'   问题: {test_question}')
    print(f'   回答: {response}')
    print(f'   Tokens: {tokens}  Latency: {latency:.2f}s')
except Exception as e:
    print(f'❌ 连通性测试失败: {e}')

✅ [deepseek] 连通性测试通过
   问题: 公司的注册资本是多少？
   回答: 1000万元人民币
   Tokens: 96  Latency: 0.80s


## Step 3：批量评测

调整下方 `MODEL_KEYS` 和 `MAX_SAMPLES` 控制评测范围：

In [ ]:
# ── 可调参数 ──────────────────────────────────
MODEL_KEYS = ['deepseek', 'kimi', 'qwen']       # 要评测的模型，可选: 'deepseek', 'kimi', 'qwen'
MAX_SAMPLES = None              # None = 跑全量（105 条/模型，共 315 次 API 调用）
RESUME = True                   # True = 断点续跑，跳过已有结果
# ────────────────────────────────────────────

dataset_path = '../data/processed/niah_dataset.jsonl'

if not Path(dataset_path).exists():
    print('⚠️  数据集不存在，请先运行 01_data_preparation.ipynb')
else:
    df = run_eval(
        dataset_path=dataset_path,
        model_keys=MODEL_KEYS,
        config=config,
        output_dir='../results/raw',
        max_samples=MAX_SAMPLES,
        resume=RESUME,
    )
    print(f'\n完成！共 {len(df)} 条结果')
    df.head(5)

📂 加载 20 条样本，来自: ../data/processed/niah_dataset.jsonl
   已有 20 条结果，启用断点续跑

🚀 [deepseek] deepseek-chat — RPM 限制: 60


deepseek: 100%|██████████| 20/20 [00:00<00:00, 181571.60req/s]



🚀 [kimi] moonshot-v1-128k — RPM 限制: 20


kimi:  35%|███▌      | 7/20 [00:28<00:53,  4.11s/req]

  ⚠️  API 调用失败（尝试 1/3）: Error code: 400 - {'error': {'code': 400, 'message': 'The request was rejected because it was considered high risk', 'param': 'prompt', 'type': 'content_filter'}}
  ⚠️  API 调用失败（尝试 2/3）: Error code: 400 - {'error': {'code': 400, 'message': 'The request was rejected because it was considered high risk', 'param': 'prompt', 'type': 'content_filter'}}
  ⚠️  API 调用失败（尝试 3/3）: Error code: 400 - {'error': {'code': 400, 'message': 'The request was rejected because it was considered high risk', 'param': 'prompt', 'type': 'content_filter'}}


...

  ⚠️  API 调用失败（尝试 1/3）: Error code: 400 - {'error': {'message': 'Input data may contain inappropriate content. For details, see: https://help.aliyun.com/zh/model-studio/error-code#inappropriate-content', 'type': 'data_inspection_failed', 'param': None, 'code': 'data_inspection_failed'}, 'id': 'chatcmpl-10bee957-5bb5-9c92-957b-520dfb39ea5e', 'request_id': '10bee957-5bb5-9c92-957b-520dfb39ea5e'}
  ⚠️  API 调用失败（尝试 2/3）: Error code: 400 - {'error': {'message': 'Input data may contain inappropriate content. For details, see: https://help.aliyun.com/zh/model-studio/error-code#inappropriate-content', 'type': 'data_inspection_failed', 'param': None, 'code': 'data_inspection_failed'}, 'id': 'chatcmpl-a7ffe7ea-2139-9703-8cf0-5a619459a1cb', 'request_id': 'a7ffe7ea-2139-9703-8cf0-5a619459a1cb'}
  ⚠️  API 调用失败（尝试 3/3）: Error code: 400 - {'error': {'message': 'Input data may contain inappropriate content. For details, see: https://help.aliyun.com/zh/model-studio/error-code#inappropriate-content', 't

qwen:  95%|█████████▌| 19/20 [00:47<00:02,  2.16s/req]

  ⚠️  API 调用失败（尝试 1/3）: Error code: 400 - {'error': {'message': 'Input data may contain inappropriate content. For details, see: https://help.aliyun.com/zh/model-studio/error-code#inappropriate-content', 'type': 'data_inspection_failed', 'param': None, 'code': 'data_inspection_failed'}, 'id': 'chatcmpl-757c0c26-ae37-98fc-8456-3729aff912bd', 'request_id': '757c0c26-ae37-98fc-8456-3729aff912bd'}
  ⚠️  API 调用失败（尝试 2/3）: Error code: 400 - {'error': {'message': 'Input data may contain inappropriate content. For details, see: https://help.aliyun.com/zh/model-studio/error-code#inappropriate-content', 'type': 'data_inspection_failed', 'param': None, 'code': 'data_inspection_failed'}, 'id': 'chatcmpl-01d20ca6-a507-9a79-a33d-6f4c78c444d9', 'request_id': '01d20ca6-a507-9a79-a33d-6f4c78c444d9'}
  ⚠️  API 调用失败（尝试 3/3）: Error code: 400 - {'error': {'message': 'Input data may contain inappropriate content. For details, see: https://help.aliyun.com/zh/model-studio/error-code#inappropriate-content', 't

qwen: 100%|██████████| 20/20 [01:00<00:00,  3.01s/req]


✅ 结果保存至: ../results/raw/raw_results.csv（共 60 条）

完成！共 60 条结果


## Step 4：快速预览结果

In [17]:
results_path = '../results/raw/raw_results.csv'

if Path(results_path).exists():
    df = pd.read_csv(results_path)
    print(f'结果统计：{len(df)} 条')
    print(f'\n模型分布：')
    print(df['model'].value_counts().to_string())
    print(f'\n深度分布：')
    print(df['depth_pct'].value_counts().sort_index().to_string())
    print(f'\n前 5 条样本：')
    display(df[['model', 'context_length', 'depth_pct', 'expected_answer', 'model_response', 'latency_s']].head(5))

结果统计：60 条

模型分布：
model
deepseek    20
kimi        20
qwen        20

深度分布：
depth_pct
0      9
10     9
25     9
50     9
75     9
90     9
100    6

前 5 条样本：


,model,context_length,depth_pct,expected_answer,model_response,latency_s
0,deepseek,2000,0,47.3亿元人民币,47.3亿元人民币,0.78
1,deepseek,2000,10,EMP-2024-08831,EMP-2024-08831,0.42
2,deepseek,2000,25,暗星计划，2024年11月8日,内部项目代号为“暗星计划”，正式启动日期为2024年11月8日。,0.70
3,deepseek,2000,50,47.3亿元人民币,47.3亿元人民币,0.49
4,deepseek,2000,75,47.3亿元人民币,47.3亿元人民币。,0.45


## Step 5：估算 Token 消耗 & 费用

In [18]:
if Path(results_path).exists():
    df = pd.read_csv(results_path)
    
    cost_per_m = {'deepseek': 1.0, 'kimi': 12.0, 'qwen': 4.0}  # 约 ¥/M tokens
    
    print('Token 消耗 & 费用估算：')
    for model, sub in df.groupby('model'):
        total_tokens = sub['tokens_used'].sum()
        cpm = cost_per_m.get(model, 5.0)
        cost = total_tokens / 1e6 * cpm
        print(f'  {model:12s}: {total_tokens:>10,} tokens  ≈ ¥{cost:.3f}')

Token 消耗 & 费用估算：
  deepseek    :     45,704 tokens  ≈ ¥0.046
  kimi        :     47,930 tokens  ≈ ¥0.575
  qwen        :     50,405 tokens  ≈ ¥0.202


## ✅ 评测完成

下一步：打开 `03_analysis_visualization.ipynb` 进行数据分析和可视化